In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q faiss-cpu sentence-transformers transformers

In [ ]:
import pandas as pd
import numpy as np
import faiss
import torch

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
train = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

print(train.shape)

In [ ]:
print("Creating knowledge base")

kb = []

for idx, row in train.iterrows():
    correct_letter = row["answer"]
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index")

model = SentenceTransformer("all-MiniLM-L6-v2")

kb_embeddings = model.encode(
    kb,
    show_progress_bar=False
)

index = faiss.IndexFlatL2(
    kb_embeddings.shape[1]
)

index.add(kb_embeddings)

print("Knowledge base successfully created")
print("Documents:", len(kb))

In [ ]:
zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if torch.cuda.is_available() else -1
)

row_150 = train.iloc[150]

prompt_150 = str(row_150["prompt"])

labels_150 = [
    str(row_150["A"]),
    str(row_150["B"]),
    str(row_150["C"]),
    str(row_150["D"]),
    str(row_150["E"])
]

ans_150 = str(
    row_150[row_150["answer"]]
)

In [ ]:
result = zs(
    prompt_150,
    candidate_labels=labels_150
)

correct_score = result["scores"][
    result["labels"].index(ans_150)
]

print("Correct option probability:", round(correct_score, 3))

In [ ]:
query_embedding = model.encode(
    [prompt_150]
)

distances, indices = index.search(
    query_embedding,
    10
)

retrieved_indices = indices[0]

print("Retrieved indices:", retrieved_indices)

true_rank = list(retrieved_indices).index(150) + 1

print("True document rank:", true_rank)

In [ ]:
cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

docs_10 = [
    kb[i] for i in retrieved_indices
]

pairs = [
    [prompt_150, doc]
    for doc in docs_10
]

ce_scores = cross_encoder.predict(pairs)

sorted_positions = np.argsort(
    ce_scores
)[::-1]

reranked_indices = [
    retrieved_indices[i]
    for i in sorted_positions
]

print("Reranked indices:", reranked_indices)

true_rank = reranked_indices.index(150) + 1

print("True document rank:", true_rank)

In [ ]:
row_42 = train.iloc[42]

prompt_42 = str(row_42["prompt"])

query_embedding = model.encode(
    [prompt_42]
)

distances, indices_42 = index.search(
    query_embedding,
    5
)

docs_5 = [
    kb[i]
    for i in indices_42[0]
]

context = " ".join(docs_5)

rag_text = (
    f"Context: {context} "
    f"Question: {prompt_42}"
)

bert_tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

tokens = bert_tokenizer(
    rag_text,
    truncation=False
)

print("Total tokens:", len(tokens["input_ids"]))

In [ ]:
true_document = kb[150]

rag_text = (
    f"Context: {true_document} "
    f"Question: {prompt_150}"
)

result = zs(
    rag_text,
    candidate_labels=labels_150
)

correct_score = result["scores"][
    result["labels"].index(ans_150)
]

print("Correct option probability:", round(correct_score, 3))

In [ ]:
wrong_document = kb[999]

bad_rag_text = (
    f"Context: {wrong_document} "
    f"Question: {prompt_150}"
)

result = zs(
    bad_rag_text,
    candidate_labels=labels_150
)

correct_score = result["scores"][
    result["labels"].index(ans_150)
]

print("Correct option probability:", round(correct_score, 3))

In [ ]:
hits = 0

for i in range(100):

    row = train.iloc[i]

    prompt = str(row["prompt"])
    correct_doc = str(row[row["answer"]])

    query_embedding = model.encode(
        [prompt]
    )

    distances, retrieved = index.search(
        query_embedding,
        5
    )

    docs = [
        kb[j]
        for j in retrieved[0]
    ]

    if any(correct_doc in doc for doc in docs):
        hits += 1

hit_rate = (hits / 100) * 100

print("Hits:", hits)
print("Hit Rate:", round(hit_rate, 1), "%")

In [ ]:
def map_at_3(actual, predictions):

    predictions = predictions[:3]

    if actual not in predictions:
        return 0.0

    rank = predictions.index(actual) + 1

    return 1 / rank

In [ ]:
letters = ["A", "B", "C", "D", "E"]

map_scores = []

for i in range(20):

    row = train.iloc[i]

    prompt = str(row["prompt"])
    actual = str(row["answer"])

    # Retrieve
    query_embedding = model.encode(
        [prompt]
    )

    distances, retrieved = index.search(
        query_embedding,
        5
    )

    retrieved_indices = retrieved[0]

    docs = [
        kb[j]
        for j in retrieved_indices
    ]

    # Rerank
    pairs = [
        [prompt, doc]
        for doc in docs
    ]

    ce_scores = cross_encoder.predict(pairs)

    best_position = np.argmax(ce_scores)

    best_document = docs[best_position]

    # Add context
    rag_text = (
        f"Context: {best_document} "
        f"Question: {prompt}"
    )

    labels = [
        str(row[letter])
        for letter in letters
    ]

    # Predict
    result = zs(
        rag_text,
        candidate_labels=labels
    )

    label_to_letter = {
        str(row[letter]): letter
        for letter in letters
    }

    ranked_letters = [
        label_to_letter[label]
        for label in result["labels"]
    ]

    top3 = ranked_letters[:3]

    score = map_at_3(
        actual,
        top3
    )

    map_scores.append(score)

    print(i, actual, top3, score)


final_map3 = np.mean(map_scores)

print(
    "\nFinal MAP@3:",
    round(final_map3, 3)
)